# Plot runs from `runs/`

Reads every `runs/*/config.json` + `metrics.csv`. Filter with config keys, e.g. `load_runs(dataset="mouse", training_mode="from_scratch")`; a filter value can also be a function. Metric throughout: mean per-neuron validation correlation. Figures are also saved to `research/run_figures/`.

Sensorium runs made before the withheld-trial fix (no `response_filter="real_only"` in their config) trained and validated on all-zero test-tier trials, and mouse transfers that used one of them as the backbone inherit that. They are hidden unless `SHOW_CONTAMINATED = True`.

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

from mouse_model.run_logging import load_runs, plot_runs, summary_table

FIG_DIR = os.path.join("research", "run_figures")
os.makedirs(FIG_DIR, exist_ok=True)
COLORS = {"mouse": "C0", "sensorium": "C1"}

SHOW_CONTAMINATED = False

_all_runs = [r for r in load_runs() if r["summary"]]  # finished runs only
_by_name = {os.path.basename(r["dir"]): r for r in _all_runs}


def contaminated(r):
    """Sensorium run without the withheld-trial filter, or a transfer whose backbone came from one."""
    c = r["config"]
    if c.get("dataset") == "sensorium" and c.get("response_filter") != "real_only":
        return True
    src = _by_name.get(c.get("pretrained_run") or "")
    return src is not None and contaminated(src)


runs = [r for r in _all_runs if SHOW_CONTAMINATED or not contaminated(r)]
print(f"{len(_all_runs) - len(runs)} contaminated runs hidden")
print(len(runs), "finished runs")
for row in summary_table(runs, keys=("dataset", "session", "training_mode", "train_minutes_requested", "seed")):
    print(row)

DEFAULT_SESSION = {"mouse": "070921_J553RT", "sensorium": "dynamic29515-10-12-Video-9b4f6a1a067fe51e15306b9628efea20"}


def short(session):
    return session.split("-")[0].replace("dynamic", "") if session.startswith("dynamic") else session.split("_")[-1]


def select(dataset, mode="from_scratch", session="default", seed=0):
    """session="default" -> the original animal, None -> all animals; seed=None -> all seeds."""
    if session == "default":
        session = DEFAULT_SESSION[dataset]
    return [r for r in runs if r["config"].get("dataset") == dataset and r["config"].get("training_mode") == mode
            and (session is None or r["config"].get("session") == session)
            and (seed is None or r["config"].get("seed") == seed)]


def best_cor(r):
    return r["summary"]["val_cor_at_best"]

## Validation correlation vs epoch, one panel per dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, dataset in zip(axes, ["mouse", "sensorium"]):
    sel = sorted(select(dataset), key=lambda r: r["config"]["split_info"]["train_minutes"])
    for r in sel:
        m = r["metrics"]
        ax.plot(m["epoch"], m["val_cor"], label=f"{r['config']['split_info']['train_minutes']:.1f} min")
    ax.set_title(f"{dataset}: from scratch")
    ax.set_xlabel("epoch (>= 100 batches each)")
    ax.grid(alpha=0.3)
    ax.legend(title="training data", fontsize=8)
axes[0].set_ylabel("val correlation")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "val_cor_vs_epoch.png"), dpi=120)

## Learning curve: correlation vs minutes of training data, across animals

Thin lines: one animal each (mouse J553RT is averaged over its seeds). Thick: mean ± SEM across animals at each requested size ("full" is plotted at the mean actual minutes).
Saturating fit \(r(m) = r_{max}\, m / (m + m_{50})\) on all per-animal points; \(N_{90} = 9\, m_{50}\).
Right panel divides each animal's curve by its own full-data correlation, which removes per-animal differences in neuron count and recording quality.


In [ ]:
def sat(m, rmax, m50):
    return rmax * m / (m + m50)


def per_animal_curves(dataset):
    """{session: [(requested, minutes, mean cor over seeds, n seeds)]} for scratch runs."""
    out = {}
    for r in select(dataset, session=None, seed=None):
        c = r["config"]
        out.setdefault(c["session"], {}).setdefault(c["train_minutes_requested"], []).append(
            (c["split_info"]["train_minutes"], best_cor(r)))
    curves = {}
    for sess, by_req in out.items():
        rows = [(req, np.mean([m for m, _ in v]), np.mean([y for _, y in v]), len(v)) for req, v in by_req.items()]
        curves[sess] = sorted(rows, key=lambda t: t[1])
    return curves


fig, (ax, axn) = plt.subplots(1, 2, figsize=(14, 5))
fits, table = {}, []
for dataset in ["mouse", "sensorium"]:
    curves = per_animal_curves(dataset)
    if not curves:
        continue
    col = COLORS[dataset]
    allx, ally, by_req, by_req_n = [], [], {}, {}
    for sess, rows in curves.items():
        x = np.array([t[1] for t in rows]); y = np.array([t[2] for t in rows])
        full = [t[2] for t in rows if t[0] is None]
        ax.plot(x, y, "o-", color=col, alpha=0.25, lw=1, ms=3)
        if full:
            axn.plot(x, y / full[0], "o-", color=col, alpha=0.25, lw=1, ms=3)
        allx += list(x); ally += list(y)
        for req, m, yy, _ in rows:
            by_req.setdefault(req, []).append((m, yy))
            if full:
                by_req_n.setdefault(req, []).append((m, yy / full[0]))
    for target, d in [(ax, by_req), (axn, by_req_n)]:
        reqs = sorted(d, key=lambda q: np.mean([m for m, _ in d[q]]))
        mx = np.array([np.mean([m for m, _ in d[q]]) for q in reqs])
        my = np.array([np.mean([v for _, v in d[q]]) for q in reqs])
        se = np.array([np.std([v for _, v in d[q]], ddof=1) / np.sqrt(len(d[q])) if len(d[q]) > 1 else 0 for q in reqs])
        target.errorbar(mx, my, yerr=se, fmt="o-", color=col, lw=2.5, capsize=3,
                        label=f"{dataset} (n={len(curves)} animals)")
    for q in sorted(by_req, key=lambda q: np.mean([m for m, _ in by_req[q]])):
        v = [y for _, y in by_req[q]]
        table.append((dataset, "full" if q is None else q, len(v), np.mean(v), np.std(v, ddof=1) if len(v) > 1 else 0))
    allx, ally = np.array(allx), np.array(ally)
    if len(allx) >= 3:
        (rmax, m50), _ = curve_fit(sat, allx, ally, p0=[ally.max(), np.median(allx)], bounds=([0, 1e-3], [1, 1e4]))
        fits[dataset] = dict(rmax=round(rmax, 3), m50=round(m50, 2), n90=round(9 * m50, 1))
        xs = np.geomspace(allx.min() * 0.8, allx.max() * 1.2, 200)
        ax.plot(xs, sat(xs, rmax, m50), "--", color=col, alpha=0.8,
                label=f"fit: r_max={rmax:.3f}, N90={9 * m50:.0f} min")
for a in (ax, axn):
    a.set_xscale("log")
    a.set_xlabel("training data (minutes of recording)")
    a.grid(alpha=0.3, which="both")
    a.legend(fontsize=8)
ax.set_ylabel("val correlation at selected epoch")
ax.set_title("Data efficiency: free-moving mouse vs Sensorium (same CNN, scratch)")
axn.set_ylabel("fraction of the animal's full-data correlation")
axn.axhline(0.9, color="gray", ls=":")
axn.set_title("Normalized per animal")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "learning_curve_minutes.png"), dpi=120)
print(fits)
for row in table:
    print(f"{row[0]:9s} {str(row[1]):>5s}  n={row[2]}  cor {row[3]:.3f} ± {row[4]:.3f} (sd)")


## Seed-to-seed spread (mouse J553RT)


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
by_seed = {}
for r in select("mouse", seed=None):
    by_seed.setdefault(r["config"]["seed"], []).append((r["config"]["split_info"]["train_minutes"], best_cor(r)))
for seed, pts in sorted(by_seed.items()):
    pts.sort()
    ax.plot([m for m, _ in pts], [y for _, y in pts], "o-", label=f"seed {seed}")
ax.set_xscale("log")
ax.set_xlabel("training data (minutes)")
ax.set_ylabel("val correlation at selected epoch")
ax.set_title("Mouse J553RT: seeds change the split and the init")
ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "mouse_seed_spread.png"), dpi=120)


## Same curves vs gradient steps (separates data size from number of updates)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, dataset in zip(axes, ["mouse", "sensorium"]):
    for r in sorted(select(dataset), key=lambda r: r["config"]["split_info"]["train_minutes"]):
        m = r["metrics"]
        ax.plot(m["step"], m["val_cor"], label=f"{r['config']['split_info']['train_minutes']:.1f} min")
    ax.set_xscale("log")
    ax.set_xlabel("gradient steps")
    ax.set_title(dataset)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel("val correlation")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "val_cor_vs_steps.png"), dpi=120)

## Transfer vs scratch (full training data)

In [ ]:
MODES = [("from_scratch", "scratch"), ("transfer_frozen_backbone", "frozen backbone"),
         ("transfer_finetune", "fine-tune")]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (target, source) in zip(axes, [("mouse", "sensorium"), ("sensorium", "mouse")]):
    for i, (mode, label) in enumerate(MODES):
        sel = [r for r in select(target, mode) if r["config"].get("train_minutes_requested") is None]
        if not sel:
            continue
        r = sel[-1]
        m = r["metrics"]
        name = label if mode == "from_scratch" else f"{label} (from {source})"
        ax.plot(m["epoch"], m["val_cor"], color=f"C{i}", label=f"{name}: {best_cor(r):.3f}")
    ax.set_title(f"target: {target}")
    ax.set_xlabel("epoch")
    ax.set_ylabel("val correlation")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "transfer_vs_scratch.png"), dpi=120)

## Per-neuron correlation at the best epoch (full-data runs)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for r in runs:
    p = os.path.join(r["dir"], "per_neuron_best.npz")
    cfg = r["config"]
    if cfg.get("train_minutes_requested") is None and cfg.get("training_mode") == "from_scratch" and os.path.isfile(p):
        cor = np.load(p)["cor_per_neuron"]
        cor = cor[np.isfinite(cor)]
        ax.hist(cor, bins=40, alpha=0.5, density=True, color=COLORS[cfg["dataset"]],
                label=f"{cfg['dataset']} (n={len(cor)}, median {np.median(cor):.3f})")
ax.set_xlabel("per-neuron val correlation")
ax.set_ylabel("density")
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "per_neuron_correlation.png"), dpi=120)

## Sensorium before vs after the input fix

Old curve: legacy pickle (image rows across time as "frames", random frame-level split). New: real frames, trial-level split.

In [ ]:
old_path = "epoch_vs_correlation_data/epoch_vs_score_cnn_sensorium_random70_perframe_shifter_False.pkl"
fig, ax = plt.subplots(figsize=(7.5, 4.5))
if os.path.isfile(old_path):
    old = pickle.load(open(old_path, "rb"))
    old_train_n = 17917  # old random 70/30 split of 711 trials x 36 'frames'
    ax.plot((np.arange(len(old["val_cor_list"])) + 1) * old_train_n, old["val_cor_list"], "--", color="gray",
            label="old: wrong axis + frame-level split")
new = [r for r in select("sensorium") if r["config"].get("train_minutes_requested") is None]
if new:
    m = new[-1]["metrics"]
    ax.plot(m["samples_seen"], m["val_cor"], color=COLORS["sensorium"], label="new: real frames + trial-level split")
ax.set_xscale("log")
ax.set_xlabel("training samples seen")
ax.set_ylabel("val correlation")
ax.set_title("Sensorium CNN (scratch, full data)")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "sensorium_before_after_fix.png"), dpi=120)

## Pooled multi-readout vs single-animal (full data)

Per-session validation correlation of the shared-core model next to the single-animal full-data run with the same seed (same train/val split per session).

In [ ]:
pooled = [r for r in runs if r["config"].get("training_mode") == "pooled_multi_readout"]
if not pooled:
    print("no pooled runs yet")
for r in pooled:
    c = r["config"]
    per = r["summary"].get("val_cor_per_session_at_best") or {}
    print(f"{c['dataset']} pooled over {len(c['sessions'])} sessions ({os.path.basename(r['dir'])}): "
          f"macro val cor {best_cor(r):.3f}")
    diffs = []
    for k, sess in enumerate(c["sessions"]):
        single = [best_cor(x) for x in select(c["dataset"], session=sess, seed=c["seed"])
                  if x["config"].get("train_minutes_requested") is None]
        p = per.get(str(k), float("nan"))
        if single:
            diffs.append(p - single[-1])
        print(f"  {short(sess):>8s}  pooled {p:.3f}  single {single[-1] if single else float('nan'):.3f}")
    if diffs:
        print(f"  mean pooled - single: {np.mean(diffs):+.3f}")